# Previsão de Preços de Passagens Aéreas com Big Data

Pipeline em PySpark sobre o dataset Expedia (~82M registros, 16 aeroportos).
A estrutura segue [`docs/proposta-projeto.md`](../docs/proposta-projeto.md):

1. Setup
2. Carregamento do dataset
3. Pré-processamento (proposta §5.1)
4. Engenharia de atributos (proposta §5.2)
5. Análise exploratória (proposta §5.3)
6. Modelagem (proposta §6)
7. Conclusões


## 1. Setup


### 1.1 Imports


In [ ]:
import os
import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from pyspark.sql import SparkSession
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.feature import OneHotEncoder, StringIndexer, VectorAssembler
from pyspark.ml.regression import DecisionTreeRegressor, LinearRegression

plt.rcParams["figure.dpi"] = 100
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.3


### 1.2 Sessão Spark e helpers

`get_or_create_spark` configura shuffle partitions baixo (16) por causa do cluster
de 1 worker / 2 cores e habilita eager-eval para previews legíveis. `run_sql`
encapsula o padrão `display(df.limit(N).toPandas())` usado em todo o notebook.


In [ ]:
APP_NAME = "flight-price-sql-mllib"
SPARK_MASTER_URL = os.environ.get("SPARK_MASTER", "spark://spark-master:7077")


def get_or_create_spark(app_name: str = APP_NAME) -> SparkSession:
    spark_session = (
        SparkSession.builder
        .appName(app_name)
        .master(SPARK_MASTER_URL)
        .config("spark.sql.shuffle.partitions", "16")
        .config("spark.sql.session.timeZone", "UTC")
        .config("spark.sql.legacy.timeParserPolicy", "LEGACY")
        .config("spark.sql.repl.eagerEval.enabled", "true")
        .config("spark.sql.repl.eagerEval.maxNumRows", "20")
        .config("spark.sql.repl.eagerEval.truncate", "80")
        .getOrCreate()
    )
    spark_session.sparkContext.setLogLevel("WARN")
    return spark_session


def run_sql(query: str, preview_rows: int = 20):
    df = spark.sql(query)
    display(df.limit(preview_rows).toPandas())
    return df


## 2. Carregamento do Dataset


### 2.1 Caminhos do dataset

O Parquet (~6.8 GB, snappy, particionado por `startingAirport`) é gerado por
`make convert-local` na raiz do projeto. `LOAD_SAMPLE_RATIO` controla a fração
do dataset usada — 0.1 (≈8M linhas) é o padrão para iteração rápida; subir para
1.0 para benchmarks finais.


In [ ]:
HDFS_PARQUET_PATH  = "hdfs://namenode:9000/data/itineraries.parquet"
LOCAL_PARQUET_PATH = "/data/itineraries.parquet"

LOAD_SAMPLE_RATIO = 0.1
DEV_SAMPLE_RATIO = 1.0


def resolve_parquet_paths(spark_session: SparkSession) -> list[str]:
    if spark_session.sparkContext.master.startswith("local"):
        return [HDFS_PARQUET_PATH, LOCAL_PARQUET_PATH]
    return [HDFS_PARQUET_PATH]


def load_flights(spark_session: SparkSession, candidate_paths: list[str]):
    last_error = None
    print(f"Tentando carregar Parquet pelos caminhos: {candidate_paths}")
    for path in candidate_paths:
        try:
            df = spark_session.read.parquet(path)
            print(f"Dataset registrado: {path}")
            return df, path
        except Exception as exc:
            print(f"Falha ao carregar {path}: {exc}")
            last_error = exc
    raise RuntimeError(
        "Nao foi possivel carregar o dataset Parquet. "
        "Rode `make convert-local` no projeto para gera-lo."
    ) from last_error


def apply_load_sample(df, ratio: float):
    if ratio >= 1.0:
        print("Usando dataset completo apos a leitura.")
        return df
    print(f"Aplicando amostra de carga com ratio={ratio} para acelerar as iteracoes.")
    return df.sample(withReplacement=False, fraction=ratio, seed=42)


### 2.2 Leitura


In [ ]:
spark = get_or_create_spark()
candidate_paths = resolve_parquet_paths(spark)
raw_input_df, source_path = load_flights(spark, candidate_paths)
raw_df = apply_load_sample(raw_input_df, LOAD_SAMPLE_RATIO)
raw_df.createOrReplaceTempView("flights_raw")

print("Spark version:", spark.version)
print("Spark master:", spark.sparkContext.master)
print("Source path:", source_path)
print("Load sample ratio:", LOAD_SAMPLE_RATIO)


### 2.3 Amostra dos dados brutos


In [ ]:
run_sql("SELECT * FROM flights_raw LIMIT 5")


## 3. Pré-processamento (proposta §5.1)

Limpeza e conversão de tipo. Sem features derivadas ainda — essas vão para a §4.
Produz o view `flights_typed` com colunas tipadas (datas como `DATE`, booleans
como `0/1`, numéricos como `DOUBLE`/`INT`) e arrays parseados das colunas
pipe-delimited (`segments*`).


### 3.1 Conversão de tipos e parsing dos segmentos

- `searchDate`/`flightDate` → `DATE`
- `isBasicEconomy`/`isRefundable`/`isNonStop` → `INT (0/1)`
- `baseFare`/`totalFare`/`totalTravelDistance` → `DOUBLE`
- `seatsRemaining`/`elapsedDays` → `INT`
- `travelDuration` ISO-8601 → minutos (`travel_duration_minutes`)
- Colunas `segments*` (pipe-delimited `||`) → arrays via `SPLIT`+`FILTER`

`fare_basis_prefix` extrai o primeiro caractere do `fareBasisCode` (proxy para
classe tarifária — usado na EDA depois).


In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW flights_typed AS
SELECT
    legId AS leg_id,
    TO_DATE(searchDate) AS search_date,
    TO_DATE(flightDate) AS flight_date,
    startingAirport AS starting_airport,
    destinationAirport AS destination_airport,
    fareBasisCode AS fare_basis_code,
    UPPER(SUBSTRING(COALESCE(fareBasisCode, 'UNK'), 1, 1)) AS fare_basis_prefix,
    travelDuration AS travel_duration_iso,
    CAST(elapsedDays AS INT) AS overnight_days,
    CASE WHEN LOWER(CAST(isBasicEconomy AS STRING)) = 'true' THEN 1 ELSE 0 END AS is_basic_economy,
    CASE WHEN LOWER(CAST(isRefundable  AS STRING)) = 'true' THEN 1 ELSE 0 END AS is_refundable,
    CASE WHEN LOWER(CAST(isNonStop     AS STRING)) = 'true' THEN 1 ELSE 0 END AS is_non_stop,
    CAST(baseFare  AS DOUBLE) AS base_fare,
    CAST(totalFare AS DOUBLE) AS total_fare,
    CAST(seatsRemaining AS INT) AS seats_remaining,
    CAST(totalTravelDistance AS DOUBLE) AS total_travel_distance,
    (
        CAST(CASE WHEN REGEXP_EXTRACT(travelDuration, '([0-9]+)D', 1) = '' THEN '0' ELSE REGEXP_EXTRACT(travelDuration, '([0-9]+)D', 1) END AS INT) * 1440
      + CAST(CASE WHEN REGEXP_EXTRACT(travelDuration, '([0-9]+)H', 1) = '' THEN '0' ELSE REGEXP_EXTRACT(travelDuration, '([0-9]+)H', 1) END AS INT) * 60
      + CAST(CASE WHEN REGEXP_EXTRACT(travelDuration, '([0-9]+)M', 1) = '' THEN '0' ELSE REGEXP_EXTRACT(travelDuration, '([0-9]+)M', 1) END AS INT)
    ) AS travel_duration_minutes,
    CAST(NULLIF(REGEXP_EXTRACT(COALESCE(segmentsDepartureTimeRaw, ''), 'T([0-9]{2}):', 1), '') AS INT) AS departure_hour,
    COALESCE(segmentsDepartureAirportCode, '') AS segments_departure_airport_code_raw,
    FILTER(SPLIT(REPLACE(COALESCE(segmentsDepartureAirportCode, ''), '||', '~'), '~'), x -> TRIM(x) <> '') AS departure_airport_segments,
    FILTER(SPLIT(REPLACE(COALESCE(segmentsDurationInSeconds,    ''), '||', '~'), '~'), x -> TRIM(x) <> '') AS duration_seconds_segments,
    FILTER(SPLIT(REPLACE(COALESCE(segmentsDistance,             ''), '||', '~'), '~'), x -> TRIM(x) <> '' AND LOWER(TRIM(x)) <> 'none') AS distance_segments,
    FILTER(SPLIT(REPLACE(COALESCE(segmentsAirlineCode,          ''), '||', '~'), '~'), x -> TRIM(x) <> '') AS airline_segments,
    FILTER(SPLIT(REPLACE(COALESCE(segmentsCabinCode,            ''), '||', '~'), '~'), x -> TRIM(x) <> '') AS cabin_segments
FROM flights_raw
WHERE totalFare IS NOT NULL
""")


### 3.2 Verificação rápida do schema tipado


In [ ]:
run_sql("SELECT * FROM flights_typed LIMIT 5")


## 4. Engenharia de Atributos (proposta §5.2)

Sobre `flights_typed`, derivamos features que entram tanto na EDA quanto no
modelo. Saída: view `flights_clean` com colunas calendário, contagem de escalas,
estatísticas de segmento, etc. Em seguida `flights_dev` aplica `DEV_SAMPLE_RATIO`
para iteração rápida (mantém-se 1.0 por padrão).

Atributos criados:

| Categoria | Colunas |
|---|---|
| Temporais   | `days_until_flight`, `flight_day_of_week`, `flight_month`, `is_weekend` |
| Rota        | `route` (origem-destino) |
| Escalas     | `segment_count`, `stop_count`, `layover_minutes`, `travel_minus_segment_minutes` |
| Distância   | `distance_per_segment`, `effective_distance` (fallback p/ NaN), `has_missing_segment_distance` |
| Diversidade | `distinct_airline_count`, `distinct_cabin_count` |
| Duração     | `segment_duration_sum_minutes`, `segment_duration_avg_minutes` |


In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW flights_clean AS
WITH prepared AS (
    SELECT
        *,
        DATEDIFF(flight_date, search_date) AS days_until_flight,
        DAYOFWEEK(flight_date) AS flight_day_of_week,
        MONTH(flight_date) AS flight_month,
        CONCAT(starting_airport, '-', destination_airport) AS route,
        CASE
            WHEN segments_departure_airport_code_raw IS NULL OR TRIM(segments_departure_airport_code_raw) = '' THEN 0
            ELSE CAST(((LENGTH(segments_departure_airport_code_raw) - LENGTH(REPLACE(segments_departure_airport_code_raw, '||', ''))) / 2) + 1 AS INT)
        END AS segment_count,
        AGGREGATE(duration_seconds_segments, CAST(0.0 AS DOUBLE), (acc, x) -> acc + COALESCE(CAST(x AS DOUBLE), 0.0D)) / 60.0 AS segment_duration_sum_minutes,
        AGGREGATE(distance_segments,         CAST(0.0 AS DOUBLE), (acc, x) -> acc + COALESCE(CAST(x AS DOUBLE), 0.0D)) AS known_segment_distance,
        SIZE(distance_segments) AS known_segment_distance_count,
        SIZE(ARRAY_DISTINCT(airline_segments)) AS distinct_airline_count,
        SIZE(ARRAY_DISTINCT(cabin_segments))   AS distinct_cabin_count
    FROM flights_typed
)
SELECT
    leg_id,
    search_date,
    flight_date,
    starting_airport,
    destination_airport,
    fare_basis_code,
    fare_basis_prefix,
    travel_duration_iso,
    overnight_days,
    days_until_flight,
    is_basic_economy,
    is_refundable,
    is_non_stop,
    base_fare,
    total_fare,
    seats_remaining,
    total_travel_distance,
    travel_duration_minutes,
    departure_hour,
    segment_count,
    GREATEST(segment_count - 1, 0) AS stop_count,
    segment_duration_sum_minutes,
    CASE WHEN segment_count > 0 THEN segment_duration_sum_minutes / segment_count END AS segment_duration_avg_minutes,
    known_segment_distance,
    known_segment_distance_count,
    CASE
        WHEN total_travel_distance IS NULL OR isnan(total_travel_distance) THEN known_segment_distance
        ELSE total_travel_distance
    END AS effective_distance,
    CASE WHEN segment_count > 0 THEN known_segment_distance / segment_count END AS distance_per_segment,
    CASE WHEN segment_count <= 1 THEN 0.0D
         ELSE GREATEST(travel_duration_minutes - segment_duration_sum_minutes, 0.0D)
    END AS layover_minutes,
    GREATEST(travel_duration_minutes - segment_duration_sum_minutes, 0.0D) AS travel_minus_segment_minutes,
    CASE WHEN segment_count > known_segment_distance_count THEN 1 ELSE 0 END AS has_missing_segment_distance,
    distinct_airline_count,
    distinct_cabin_count,
    flight_day_of_week,
    flight_month,
    CASE WHEN DAYOFWEEK(flight_date) IN (1, 7) THEN 1 ELSE 0 END AS is_weekend,
    route
FROM prepared
""")

if DEV_SAMPLE_RATIO < 1.0:
    spark.sql(f"""
    CREATE OR REPLACE TEMP VIEW flights_dev AS
    SELECT * FROM flights_clean WHERE rand(42) <= {DEV_SAMPLE_RATIO}
    """)
else:
    spark.sql("CREATE OR REPLACE TEMP VIEW flights_dev AS SELECT * FROM flights_clean")

# Cache: flights_dev e reusada em todas as queries de EDA e modelagem.
spark.table("flights_dev").cache().count()


### 4.2 Validação de qualidade dos atributos

Conta linhas com sinais de inconsistência. Tudo zero é o que esperamos —
qualquer divergência aqui justifica refinar a §3 ou §4.


In [ ]:
run_sql("""
SELECT
    COUNT(*)                                                                                AS total_rows,
    SUM(CASE WHEN days_until_flight       <  0    THEN 1 ELSE 0 END)                        AS invalid_days_until_flight,
    SUM(CASE WHEN segment_count           <= 0    THEN 1 ELSE 0 END)                        AS invalid_segment_count,
    SUM(CASE WHEN segment_count           >  4    THEN 1 ELSE 0 END)                        AS invalid_segment_upper_bound,
    SUM(CASE WHEN travel_duration_minutes <= 0    THEN 1 ELSE 0 END)                        AS invalid_travel_duration,
    SUM(CASE WHEN departure_hour IS NOT NULL AND (departure_hour < 0 OR departure_hour > 23)
             THEN 1 ELSE 0 END)                                                              AS invalid_departure_hour,
    SUM(CASE WHEN layover_minutes < 0     THEN 1 ELSE 0 END)                                AS negative_layover_signal,
    SUM(CASE WHEN is_non_stop = 1 AND stop_count <> 0      THEN 1 ELSE 0 END)               AS invalid_non_stop_stop_count,
    SUM(CASE WHEN is_non_stop = 1 AND layover_minutes > 15 THEN 1 ELSE 0 END)               AS non_stop_with_layover,
    MIN(total_fare) AS min_total_fare,
    MAX(total_fare) AS max_total_fare,
    ROUND(AVG(total_fare),               2) AS avg_total_fare,
    ROUND(AVG(travel_duration_minutes),  2) AS avg_duration_minutes,
    ROUND(AVG(days_until_flight),        2) AS avg_days_until_flight,
    ROUND(AVG(layover_minutes),          2) AS avg_layover_minutes
FROM flights_dev
""")


## 5. Análise Exploratória (proposta §5.3)

Cada subseção responde a uma das perguntas listadas em §5.3 da proposta.
Padrão: a agregação é feita no Spark (escala) e só o resultado pequeno
é puxado para o driver via `.toPandas()` para plot.


### 5.1 Distribuição da variável alvo (`total_fare`)

p99 ≈ \$897 e o máximo passa de \$7k — cauda longa. O histograma é cortado em
\$1000 para legibilidade; a tabela mostra os percentis sem corte.


In [ ]:
_clip = 1000
_pdf = spark.sql(f"""
SELECT FLOOR(LEAST(total_fare, {_clip}) / 25) * 25 AS bin_start, COUNT(*) AS n
FROM flights_dev
WHERE total_fare > 0
GROUP BY bin_start
ORDER BY bin_start
""").toPandas()

_stats = spark.sql("""
SELECT
    ROUND(MIN(total_fare),                 2) AS min,
    ROUND(percentile_approx(total_fare, 0.25), 2) AS p25,
    ROUND(percentile_approx(total_fare, 0.50), 2) AS p50,
    ROUND(AVG(total_fare),                 2) AS mean,
    ROUND(percentile_approx(total_fare, 0.95), 2) AS p95,
    ROUND(percentile_approx(total_fare, 0.99), 2) AS p99,
    ROUND(MAX(total_fare),                 2) AS max
FROM flights_dev
""").toPandas()

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(_pdf["bin_start"], _pdf["n"], width=24, align="edge", color="#3b82f6")
ax.axvline(_stats["p50"][0], color="#ef4444", linestyle="--", label=f"mediana ${_stats['p50'][0]:.0f}")
ax.axvline(_stats["mean"][0], color="#10b981", linestyle="--", label=f"média ${_stats['mean'][0]:.0f}")
ax.set_xlabel("total_fare (USD, cortado em $1000)")
ax.set_ylabel("voos")
ax.set_title(f"Distribuição de total_fare — n={_pdf['n'].sum():,}")
ax.legend()
plt.tight_layout(); plt.show()

display(_stats)


### 5.2 Correlação numérica com `total_fare`

Pearson de cada feature numérica contra `total_fare`. `base_fare` aparece como
checksum: `total_fare ≈ base_fare + taxas`, então a correlação é praticamente 1.
As demais correlações orientam o que entra no modelo.


In [ ]:
_num_cols = [
    "base_fare",
    "days_until_flight",
    "travel_duration_minutes",
    "segment_count",
    "stop_count",
    "layover_minutes",
    "effective_distance",
    "distance_per_segment",
    "seats_remaining",
    "is_basic_economy",
    "is_refundable",
    "is_non_stop",
    "flight_month",
    "is_weekend",
]
_corr_select = ", ".join(f"corr({c}, total_fare) AS {c}" for c in _num_cols)
_corr_pdf = spark.sql(f"SELECT {_corr_select} FROM flights_dev").toPandas().T
_corr_pdf.columns = ["corr"]
_corr_pdf = _corr_pdf.sort_values("corr", key=abs, ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
_colors = ["#ef4444" if v < 0 else "#3b82f6" for v in _corr_pdf["corr"]]
ax.barh(_corr_pdf.index, _corr_pdf["corr"], color=_colors)
ax.axvline(0, color="black", linewidth=0.5)
ax.set_xlabel("Correlação Pearson com total_fare")
ax.set_title("Correlação das features numéricas com total_fare")
plt.tight_layout(); plt.show()


### 5.3 Antecedência da compra

`days_until_flight` varia de 1 a 60 dias. Bins não-uniformes capturam a faixa
de comportamento clássico: compra de última hora vs. compra antecipada.


In [ ]:
_pdf = spark.sql("""
SELECT
    CASE
        WHEN days_until_flight BETWEEN 0  AND 3  THEN '0-3'
        WHEN days_until_flight BETWEEN 4  AND 7  THEN '4-7'
        WHEN days_until_flight BETWEEN 8  AND 14 THEN '8-14'
        WHEN days_until_flight BETWEEN 15 AND 21 THEN '15-21'
        WHEN days_until_flight BETWEEN 22 AND 30 THEN '22-30'
        WHEN days_until_flight BETWEEN 31 AND 45 THEN '31-45'
        ELSE '46+'
    END                                 AS bin,
    COUNT(*)                            AS n,
    ROUND(AVG(total_fare), 2)          AS avg_fare
FROM flights_dev
WHERE days_until_flight IS NOT NULL AND days_until_flight >= 0
GROUP BY bin
""").toPandas()
_order = ["0-3", "4-7", "8-14", "15-21", "22-30", "31-45", "46+"]
_pdf = _pdf.set_index("bin").loc[_order].reset_index()

fig, ax1 = plt.subplots(figsize=(9, 4))
ax2 = ax1.twinx()
ax1.bar(_pdf["bin"], _pdf["n"], color="#cbd5e1", label="volume")
ax2.plot(_pdf["bin"], _pdf["avg_fare"], color="#ef4444", marker="o", linewidth=2, label="avg_fare")
ax1.set_xlabel("Dias até o voo")
ax1.set_ylabel("Voos")
ax2.set_ylabel("avg total_fare (USD)")
ax1.set_title("Tarifa média por antecedência da compra")
ax2.grid(False)
plt.tight_layout(); plt.show()

display(_pdf)


### 5.4 Voos diretos vs. com escalas

Bar chart de tarifa média por número de paradas, com volume sobreposto.
Espera-se preço crescente com número de escalas, mas é útil quantificar.


In [ ]:
_pdf = spark.sql("""
SELECT stop_count, COUNT(*) AS n, ROUND(AVG(total_fare), 2) AS avg_fare
FROM flights_dev
WHERE stop_count IS NOT NULL
GROUP BY stop_count
ORDER BY stop_count
""").toPandas()

fig, ax1 = plt.subplots(figsize=(8, 4))
ax2 = ax1.twinx()
ax1.bar(_pdf["stop_count"].astype(str), _pdf["n"], color="#cbd5e1", label="volume")
ax2.plot(_pdf["stop_count"].astype(str), _pdf["avg_fare"], color="#ef4444", marker="o", linewidth=2, label="avg_fare")
for i, (n, fare) in enumerate(zip(_pdf["n"], _pdf["avg_fare"])):
    ax1.text(i, n, f"{n:,}", ha="center", va="bottom", fontsize=8, color="#64748b")
ax1.set_xlabel("Número de escalas (stop_count)")
ax1.set_ylabel("Voos")
ax2.set_ylabel("avg total_fare (USD)")
ax1.set_title("Tarifa média por número de escalas")
ax2.grid(False)
plt.tight_layout(); plt.show()


### 5.5 Sazonalidade mensal

O dataset cobre Abr–Nov 2022. A queda de tarifa de Jun → Nov é um dos sinais
mais marcantes que apareceram na exploração inicial.


In [ ]:
_pdf = spark.sql("""
SELECT flight_month AS month, COUNT(*) AS n, ROUND(AVG(total_fare), 2) AS avg_fare
FROM flights_dev
WHERE flight_month IS NOT NULL
GROUP BY flight_month
ORDER BY flight_month
""").toPandas()

fig, ax1 = plt.subplots(figsize=(9, 4))
ax2 = ax1.twinx()
ax1.bar(_pdf["month"].astype(str), _pdf["n"], color="#cbd5e1", label="volume")
ax2.plot(_pdf["month"].astype(str), _pdf["avg_fare"], color="#ef4444", marker="o", linewidth=2, label="avg_fare")
ax1.set_xlabel("Mês do voo")
ax1.set_ylabel("Voos")
ax2.set_ylabel("avg total_fare (USD)")
ax1.set_title("Sazonalidade — tarifa média e volume por mês")
ax2.grid(False)
plt.tight_layout(); plt.show()


### 5.6 Top 20 rotas mais caras (avg `total_fare`)

Apenas rotas com volume razoável (>1k voos) para evitar amostras pequenas.
OAK aparece dominando — investigar separadamente se vale modelar essa cidade.


In [ ]:
_pdf = spark.sql("""
SELECT route, COUNT(*) AS n, ROUND(AVG(total_fare), 2) AS avg_fare
FROM flights_dev
GROUP BY route
HAVING COUNT(*) >= 1000
ORDER BY avg_fare DESC
LIMIT 20
""").toPandas().iloc[::-1]  # invert for horizontal bar chart top->bottom

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(_pdf["route"], _pdf["avg_fare"], color="#3b82f6")
for i, (fare, n) in enumerate(zip(_pdf["avg_fare"], _pdf["n"])):
    ax.text(fare, i, f"  ${fare:.0f}  (n={n:,})", va="center", fontsize=8)
ax.set_xlabel("avg total_fare (USD)")
ax.set_title("Top 20 rotas mais caras (>=1000 voos)")
plt.tight_layout(); plt.show()


### 5.7 Classes tarifárias (prefixo do `fareBasisCode`)

Top 15 prefixos por volume, ordenados por tarifa média. Mostra a hierarquia
de classe: `T`/`S`/`X` (basic economy ~\$200) até `Y`/`B` (~\$600-800) e os
raros `J`/`F`/`A` (executiva/primeira, fora do recorte).


In [ ]:
_top = spark.sql("""
SELECT fare_basis_prefix
FROM flights_dev
GROUP BY fare_basis_prefix
ORDER BY COUNT(*) DESC
LIMIT 15
""").toPandas()["fare_basis_prefix"].tolist()

_pdf = spark.sql(f"""
SELECT fare_basis_prefix, COUNT(*) AS n, ROUND(AVG(total_fare), 2) AS avg_fare
FROM flights_dev
WHERE fare_basis_prefix IN ({", ".join(repr(p) for p in _top)})
GROUP BY fare_basis_prefix
ORDER BY avg_fare DESC
""").toPandas()

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(_pdf["fare_basis_prefix"], _pdf["avg_fare"], color="#3b82f6")
for i, (fare, n) in enumerate(zip(_pdf["avg_fare"], _pdf["n"])):
    ax.text(i, fare, f"n={n//1000}k", ha="center", va="bottom", fontsize=8, color="#64748b")
ax.set_xlabel("Prefixo da fare_basis_code")
ax.set_ylabel("avg total_fare (USD)")
ax.set_title("Tarifa média por classe tarifária (top 15 por volume)")
plt.tight_layout(); plt.show()


### 5.8 Sanity check de inconsistências

Última verificação antes da modelagem: linhas que falham nos invariantes
definidos na §4. Esperamos zero resultados.


In [ ]:
run_sql("""
SELECT
    route, travel_duration_iso, travel_duration_minutes,
    segment_count, stop_count, departure_hour, layover_minutes,
    days_until_flight, total_fare
FROM flights_dev
WHERE travel_duration_minutes <= 0
   OR segment_count <= 0
   OR days_until_flight < 0
   OR (departure_hour IS NOT NULL AND (departure_hour < 0 OR departure_hour > 23))
ORDER BY total_fare DESC
LIMIT 20
""")


## 6. Modelagem (proposta §6)


### 6.1 Filtros de qualidade para ML

Aplicamos restrições conservadoras sobre `flights_dev` antes do treino. Mantemos
apenas voos com `segment_count ∈ [1,4]`, durações e antecedências dentro de
faixas plausíveis, e exigimos consistência entre `is_non_stop` e `stop_count`.


In [ ]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW flights_ml_base AS
SELECT
    total_fare,
    base_fare,
    days_until_flight,
    overnight_days,
    seats_remaining,
    COALESCE(effective_distance, 0.0D) AS total_travel_distance,
    travel_duration_minutes,
    segment_count,
    stop_count,
    flight_day_of_week,
    flight_month,
    is_basic_economy,
    is_refundable,
    is_non_stop,
    route,
    flight_date,
    starting_airport,
    destination_airport,
    fare_basis_prefix,
    departure_hour,
    segment_duration_sum_minutes,
    segment_duration_avg_minutes,
    effective_distance,
    distance_per_segment,
    layover_minutes,
    has_missing_segment_distance,
    distinct_airline_count,
    distinct_cabin_count,
    is_weekend,
    travel_minus_segment_minutes
FROM flights_dev
WHERE total_fare              IS NOT NULL
  AND base_fare               IS NOT NULL
  AND days_until_flight       IS NOT NULL
  AND overnight_days          IS NOT NULL
  AND seats_remaining         IS NOT NULL
  AND travel_duration_minutes IS NOT NULL
  AND departure_hour          IS NOT NULL
  AND segment_count BETWEEN 1 AND 4
  AND departure_hour BETWEEN 0 AND 23
  AND travel_duration_minutes BETWEEN 30 AND 4320
  AND segment_duration_sum_minutes > 0
  AND days_until_flight BETWEEN 0 AND 365
  AND layover_minutes   BETWEEN 0 AND 1440
  AND (is_non_stop = 0 OR stop_count = 0)
  AND (is_non_stop = 0 OR layover_minutes <= 15)
""")

run_sql("""
SELECT
    COUNT(*)                                AS total_rows,
    MIN(flight_date)                        AS min_flight_date,
    MAX(flight_date)                        AS max_flight_date,
    ROUND(AVG(total_fare),         2)      AS avg_total_fare,
    ROUND(AVG(days_until_flight),  2)      AS avg_days_until_flight,
    ROUND(AVG(segment_count),      2)      AS avg_segment_count,
    ROUND(AVG(layover_minutes),    2)      AS avg_layover_minutes
FROM flights_ml_base
""")


### 6.2 Split temporal treino/teste

Cortamos no percentil 80 de `flight_date` — treina em datas anteriores, avalia
em datas posteriores. Reduz vazamento temporal vs. um split aleatório.


In [ ]:
cutoff_unix = spark.sql(
    "SELECT percentile_approx(unix_timestamp(flight_date), 0.8) AS cutoff_unix FROM flights_ml_base"
).first()["cutoff_unix"]

spark.sql(f"""
CREATE OR REPLACE TEMP VIEW flights_train AS
SELECT * FROM flights_ml_base WHERE unix_timestamp(flight_date) <= {cutoff_unix}
""")
spark.sql(f"""
CREATE OR REPLACE TEMP VIEW flights_test AS
SELECT * FROM flights_ml_base WHERE unix_timestamp(flight_date) >  {cutoff_unix}
""")

train_df = spark.sql("SELECT * FROM flights_train").cache()
test_df  = spark.sql("SELECT * FROM flights_test").cache()

run_sql("""
SELECT 'train' AS split_name, COUNT(*) AS total_rows,
       MIN(flight_date) AS min_flight_date, MAX(flight_date) AS max_flight_date FROM flights_train
UNION ALL
SELECT 'test'  AS split_name, COUNT(*) AS total_rows,
       MIN(flight_date) AS min_flight_date, MAX(flight_date) AS max_flight_date FROM flights_test
""")

print("Train rows:", train_df.count())
print("Test rows:",  test_df.count())


### 6.3 Pipeline de features

`StringIndexer` + `OneHotEncoder` para categóricas (`route`, aeroportos,
prefixo da tarifa) + `VectorAssembler` agregando numéricas e OHE.

Avaliamos dois cenários:
- **`with_base_fare`**: inclui `base_fare` — teto de desempenho (vaza
  praticamente toda a variável alvo, já que `total_fare ≈ base_fare + taxas`).
- **`without_base_fare`**: cenário realista para previsão antes da reserva.


In [ ]:
COMMON_FEATURE_COLS = [
    "days_until_flight", "overnight_days", "seats_remaining",
    "total_travel_distance", "travel_duration_minutes",
    "segment_count", "stop_count",
    "flight_day_of_week", "flight_month",
    "is_basic_economy", "is_refundable", "is_non_stop",
    "departure_hour",
    "segment_duration_sum_minutes", "segment_duration_avg_minutes",
    "effective_distance", "distance_per_segment",
    "layover_minutes",
    "has_missing_segment_distance",
    "distinct_airline_count", "distinct_cabin_count",
    "is_weekend",
    "travel_minus_segment_minutes",
]
CATEGORICAL_FEATURE_COLS = ["route", "starting_airport", "destination_airport", "fare_basis_prefix"]

FEATURE_SETS = {
    "with_base_fare":    ["base_fare"] + COMMON_FEATURE_COLS,
    "without_base_fare": COMMON_FEATURE_COLS,
}

regression_evaluators = {
    "rmse": RegressionEvaluator(labelCol="total_fare", predictionCol="prediction", metricName="rmse"),
    "mae":  RegressionEvaluator(labelCol="total_fare", predictionCol="prediction", metricName="mae"),
    "r2":   RegressionEvaluator(labelCol="total_fare", predictionCol="prediction", metricName="r2"),
}

train_count = train_df.count()
test_count  = test_df.count()


def build_pipeline(feature_cols, estimator):
    indexers, idx_cols, ohe_cols = [], [], []
    for column_name in CATEGORICAL_FEATURE_COLS:
        idx, ohe = f"{column_name}_index", f"{column_name}_ohe"
        indexers.append(StringIndexer(inputCol=column_name, outputCol=idx, handleInvalid="keep"))
        idx_cols.append(idx); ohe_cols.append(ohe)
    encoder   = OneHotEncoder(inputCols=idx_cols, outputCols=ohe_cols)
    assembler = VectorAssembler(inputCols=feature_cols + ohe_cols, outputCol="features")
    return Pipeline(stages=indexers + [encoder, assembler, estimator])


def train_and_evaluate(model_name, estimator, feature_set_name, feature_cols):
    pipeline = build_pipeline(feature_cols, estimator)
    fitted = pipeline.fit(train_df)
    predictions = fitted.transform(test_df)
    metrics = {
        "model": model_name,
        "feature_set": feature_set_name,
        "uses_base_fare": 1 if "base_fare" in feature_cols else 0,
        "rmse": regression_evaluators["rmse"].evaluate(predictions),
        "mae":  regression_evaluators["mae"].evaluate(predictions),
        "r2":   regression_evaluators["r2"].evaluate(predictions),
        "train_rows": train_count,
        "test_rows":  test_count,
    }
    return fitted, predictions, metrics


### 6.4 Treinamento e avaliação

Treina LR e DT em ambos os cenários (`with_base_fare` × `without_base_fare`) e
ranqueia por RMSE.


In [ ]:
model_runs = {}
results = []

for feature_set_name, feature_cols in FEATURE_SETS.items():
    linear_regression = LinearRegression(
        featuresCol="features", labelCol="total_fare", predictionCol="prediction",
        maxIter=20, regParam=0.1, elasticNetParam=0.0,
    )
    decision_tree = DecisionTreeRegressor(
        featuresCol="features", labelCol="total_fare", predictionCol="prediction",
        maxDepth=10, minInstancesPerNode=100,
    )
    for base_model_name, estimator in [
        ("linear_regression", linear_regression),
        ("decision_tree",     decision_tree),
    ]:
        run_name = f"{base_model_name}__{feature_set_name}"
        fitted, predictions, metrics = train_and_evaluate(run_name, estimator, feature_set_name, feature_cols)
        model_runs[run_name] = {"pipeline": fitted, "predictions": predictions, "metrics": metrics}
        results.append(metrics)

metrics_df = spark.createDataFrame(results)
display(metrics_df.orderBy("rmse").toPandas())


### 6.5 Análise do melhor modelo

`with_base_fare` ganha trivialmente (vazamento), então olhamos também o melhor
*sem* `base_fare` como avaliação realista. Mostramos erro médio por número de
escalas e os 20 piores casos.


In [ ]:
best_run_name         = metrics_df.orderBy("rmse").first()["model"]
best_no_base_run_name = metrics_df.filter("uses_base_fare = 0").orderBy("rmse").first()["model"]

best_predictions          = model_runs[best_run_name]["predictions"]
best_no_base_predictions  = model_runs[best_no_base_run_name]["predictions"]
best_predictions.createOrReplaceTempView("best_predictions")
best_no_base_predictions.createOrReplaceTempView("best_no_base_predictions")

print("Best overall run:",      best_run_name)
print("Best no-base-fare run:", best_no_base_run_name)

run_sql("""
SELECT
    stop_count,
    COUNT(*)                                       AS total_voos,
    ROUND(AVG(ABS(total_fare - prediction)), 2)   AS avg_absolute_error,
    ROUND(MAX(ABS(total_fare - prediction)), 2)   AS max_absolute_error,
    ROUND(AVG(total_fare),                   2)   AS avg_total_fare
FROM best_no_base_predictions
GROUP BY stop_count
ORDER BY stop_count
""")

run_sql("""
SELECT
    route, total_fare,
    ROUND(prediction,                  2) AS prediction,
    ROUND(ABS(total_fare - prediction), 2) AS absolute_error,
    days_until_flight, travel_duration_minutes, layover_minutes,
    segment_count, stop_count, is_non_stop,
    distinct_airline_count, fare_basis_prefix
FROM best_no_base_predictions
ORDER BY absolute_error DESC
LIMIT 20
""")


## 7. Conclusões e próximos passos

- Pipeline ponta-a-ponta (load → preprocess → feature engineering → EDA → ML)
  funcional sobre o Parquet em HDFS com 82M registros.
- `with_base_fare` é teto trivial; o número que importa é o RMSE do
  `without_base_fare`.

**A fazer (próximos incrementos):**

- §5: substituir queries tabulares por gráficos (histograma de `total_fare`,
  heatmap de correlação, boxplot por `stop_count`, linha de sazonalidade
  mensal, antecedência da compra).
- §6: adicionar Random Forest (proposta §6.2), gráfico comparativo de métricas,
  scatter `prediction × total_fare`, distribuição de resíduos.
- (Opcional) MLP em notebook separado — proposta §6.3.
